# Barcelona Noise Prediction: Land Use Feature Engineering
This notebook processes the official Barcelona land use shapefile and calculates the percentage of commercial, residential, industrial, green, and recreational areas within 50m and 100m buffers of our street network.

In [6]:
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import pandas as pd

print("Loading spatial data...")
# Load data (ensure your file paths match your VS Code workspace)
noise_streets = gpd.read_file("../layers/BCN_noise_streets.gpkg")
print(noise_streets.crs)  #CRS = Coordinate Reference System
print(noise_streets.shape)
print(noise_streets.columns.tolist())
print("Number of street segments:", len(noise_streets))

landuse = gpd.read_file("../layers/bcn_usossl_etrs89_shp.gpkg")
print(landuse.crs)  #CRS = Coordinate Reference System
print(landuse.shape)
print(landuse.columns.tolist())
print("Number of land use polygons:", len(landuse))

# Quick sanity check to see the loaded data
display(landuse.head(2))

Loading spatial data...
EPSG:25831
(15115, 30)
['TRAM', 'TOTAL_D', 'TOTAL_E', 'TOTAL_N', 'TOTAL_DEN', 'TRANSIT_D', 'TRANSIT_E', 'TRANSIT_N', 'TRANSIT_DEN', 'GI_TR_D', 'GI_TR_E', 'GI_TR_N', 'GI_TR_DEN', 'FFCC_D', 'FFCC_E', 'FFCC_N', 'FFCC_DEN', 'INDUST_D', 'INDUST_E', 'INDUST_N', 'INDUST_DEN', 'VIANANTS_D', 'VIANANTS_E', 'OCI_N', 'PATIS_D', 'PATIS_E', 'geometry_type', 'start', 'end', 'geometry']
Number of street segments: 15115
EPSG:25831
(14320, 14)
['FAMILIA', 'CLAU', 'District', 'NDistric', 'CBarri', 'NBarri', 'C_AEB', 'CSecCens', 'Area', 'Perimetre', 'Coord_X', 'Coord_Y', 'Descripcio', 'geometry']
Number of land use polygons: 14320


,FAMILIA,CLAU,District,NDistric,CBarri,NBarri,C_AEB,CSecCens,Area,Perimetre,Coord_X,Coord_Y,Descripcio,geometry
0,22@,13hs,10,Sant Martí,66,el Parc i la Llacuna del Poblenou,208,042,735.029735,128.008296,432836.027297,4.583821e+06,13hs - 22@,"MULTIPOLYGON (((432764.589 4583628.639, 432747..."
1,22@,22@,10,Sant Martí,66,el Parc i la Llacuna del Poblenou,208,043,5749.281523,536.159605,432603.509491,4.583503e+06,22@ - 22@,"MULTIPOLYGON (((432566.688 4583298.456, 432562..."


## Categorize Land Use and Align CRS
We map the Catalan zoning categories to our 5 machine-learning buckets. Then, we force both datasets into EPSG:25831 (ETRS89 / UTM zone 31N) so our buffer math is perfectly calculated in meters.

In [ ]:
zoning_to_ml_buckets = {
    '22@': 'commercial',
    'Casc Antic': 'residential',
    "Conservació de l'Estructura Urbana i Edificatòria": 'residential',
    'Densificació Urbana': 'residential',
    'Edificació aïllada subzones plurifamiliars': 'residential',
    'Edificació aïllada subzones unifamiliars': 'residential',
    'Habitatge dotacional i de protecció': 'residential',
    'Ordenació Volumètrica Específica': 'residential',
    'Remodelació Física': 'residential',
    'Renovació Urbana: Rehabilitació': 'residential',
    'Industrial': 'industrial',
    'Sistema de serveis tècnics': 'industrial',
    'Sistema ferroviari': 'industrial',
    'Sistema relatiu al port': 'industrial',
    'Protecció de Sistemes Generals i Vials': 'industrial',
    'Parc Forestal': 'green',
    'Parcs i Jardins Urbans': 'green',
    'Verd privat': 'green',
    "Transformació de l'ús existent (PARC)": 'green',
    'Equipaments Comunitaris i Dotacions': 'recreational',
    "Transformació de l'ús existent (EQUIPAMENT)": 'recreational' 
}

# Apply mapping
landuse['ml_category'] = landuse['FAMILIA'].map(zoning_to_ml_buckets).fillna('other')

# Align coordinate systems
noise_streets = noise_streets.to_crs(epsg=25831)
landuse = landuse.to_crs(epsg=25831)

print("CRS aligned to EPSG:25831")

KeyError: 'nom_capa'